In [ ]:
import csv
import nltk
import pandas as pd
import requests
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer

nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

url = "https://indianexpress.com/article/cities/ahmedabad/neet-re-exam-for-every-student-is-injustice-say-parents-10686452/"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}
response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text, "html.parser")
story_container = soup.find('div', id='pcl-full-content')

if story_container:
    contents = story_container.find_all('p')
else:
    contents = soup.find_all('p')

filename = "indian_express_corpus.csv"

with open(filename, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=['Article'])
    writer.writeheader()
    for content in contents:
        text = content.get_text().strip()
        if len(text) > 20:
            writer.writerow({'Article': text})

data = pd.read_csv(filename)

all_sentences = []
for block in data['Article'].dropna():
    all_sentences.extend(sent_tokenize(str(block)))

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = model.encode(all_sentences)
print(embeddings.shape)

similarities = model.similarity(embeddings, embeddings)
print(similarities)